In [ ]:
%pip install -U "weaviate-client[agents]"

In [ ]:
%pip install -U weaviate-agents

In [ ]:
import weaviate, os
from weaviate.classes.init import Auth
from weaviate.config import AdditionalConfig, Timeout
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv(override=True)

# Retrieve environment variables
CLUSTER_URL = os.getenv("CLUSTER_URL")
API_KEY = os.getenv("API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
COHERE_API_KEY = os.getenv("COHERE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Connect to Weaviate
client = weaviate.connect_to_weaviate_cloud(
	cluster_url=CLUSTER_URL,
	auth_credentials=Auth.api_key(API_KEY),
	headers={
		"X-OpenAI-Api-Key": OPENAI_API_KEY,
		"X-Cohere-Api-Key": COHERE_API_KEY,
        "X-Goog-Api-Key": GOOGLE_API_KEY
	},
	additional_config=AdditionalConfig(
		timeout=Timeout(init=30, query=60, insert=120)
	)
)

ready = client.is_ready()
server_version = client.get_meta()["version"]
client_version = weaviate.__version__
live = client.is_live()
connected = client.is_connected()

print(f"Weaviate Ready: {ready}")
print(f"Weaviate Client Version: {client_version}")
print(f"Weaviate Server Version: {server_version}")
print(f"Weaviate Live: {client.is_live()}")
print(f"Client Connected: {connected}")

In [ ]:
from weaviate.agents.query import QueryAgent

# Instantiate a new agent, and specify the collections to query
init = QueryAgent(
    client=client, collections=["<COLLECTION_NAME>"]
)

In [ ]:
# Perform a query using Ask Mode (with answer generation)
response = init.ask(
    "<ASK/QUERY>"
)

# Print the response
response.display()

In [ ]:
# Search Mode — retrieval only (no answer generation)
response = init.search(
    "<SEARCH/QUERY>",
    limit=10
)

for obj in response.search_results.objects:
    print(obj.properties)

In [ ]:
# Suggest Queries Mode — generate query ideas based on your collection's data
# Helps users discover what they can ask; great for onboarding/chatbot UX
response = init.suggest_queries(
    collections=["<COLLECTION_NAME>"],
    num_queries=3,
    instructions="<Generate query ideas that would return interesting results based on the data in the specified collection.>"
)

for suggested in response.queries:
    print(suggested.query)